In [ ]:
import sys
import pandas as pd
import shap
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.svm import SVR
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed
from sklearn.neural_network import MLPRegressor
from joblib import Memory

In [ ]:
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  
df = pd.read_pickle("/planilhas/EMB35.pkl")

____

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

[W1] - YAA_EB - Setting 4: Embeddings from all 8 sessions

In [ ]:
df_modelYAA_S4 = df.copy()
df_modelYAA_S4 = tv.standardized_delta_y(df_modelYAA_S4, MAX_HDRS, MAX_CDI)

In [ ]:
meta_colsYAA_S4 = ['Patient_ID', 'Y_Standardized_Delta_Y']
df_modelYAA_S4 = mf.get_embeddings_per_segment_mean_std_2D(df_modelYAA_S4, meta_colsYAA_S4)

In [ ]:
df_modelYAA_S4 = df_modelYAA_S4.dropna()
print(f"Patients: {df_modelYAA_S4['Patient_ID'].nunique()}")

[W2] YA_EB - Setting 4: Embeddings from all 8 sessions

In [ ]:
df_modelYA_S4 = df.copy()
df_modelYA_S4 = tv.standardized_delta_HDRS(df_modelYA_S4, MAX_HDRS)

In [ ]:
meta_colsYA_S4 = ['Patient_ID', 'Y_Standardized_Delta_HDRS']
df_modelYA_S4 = mf.get_embeddings_per_segment_mean_std_2D(df_modelYA_S4, meta_colsYA_S4)

In [ ]:
df_modelYA_S4 = df_modelYA_S4.dropna()
print(f"Patients: {df_modelYA_S4['Patient_ID'].nunique()}")

[W3] A_EB - Setting 4: Embeddings from all 8 sessions

In [ ]:
df_modelA_S4 = df.copy()
df_modelA_S4 = tv.standardized_delta_CDI(df_modelA_S4, MAX_CDI)

In [ ]:
meta_colsA_S4 = ['Patient_ID', 'Y_Standardized_Delta_CDI']
df_modelA_S4 = mf.get_embeddings_per_segment_mean_std_2D(df_modelA_S4, meta_colsA_S4)

In [ ]:
df_modelA_S4 = df_modelA_S4.dropna()
print(f"Patients: {df_modelA_S4['Patient_ID'].nunique()}")

____

Selecting the settings 

In [ ]:
datasets = [
    (df_modelYAA_S4, 'Y_Standardized_Delta_Y'),
    (df_modelYA_S4, 'Y_Standardized_Delta_HDRS'),
    (df_modelA_S4, 'Y_Standardized_Delta_CDI')
]

____

Using the Embeddings - Training the models using LOO-CV patient-independent

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()
    results_rf = Parallel(n_jobs=64, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            RandomForestRegressor(random_state=42), rf.reg_param_grid_rf
        ) for patient in unique_patients)
    print("RF Done")
    results_xgb = Parallel(n_jobs=4)(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            xgb.XGBRegressor(tree_method='gpu_hist',predictor='gpu_predictor', n_jobs=1, random_state=42), rf.reg_param_grid_xgb
        ) for patient in unique_patients)
    print("XGB Done")
    results_svr = Parallel(n_jobs=64, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            SVR(), rf.reg_param_grid_svr
        ) for patient in unique_patients)
    print("SVR Done")
    results_mlp = Parallel(n_jobs=64, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            MLPRegressor(random_state=42), rf.reg_param_grid_mlp
        ) for patient in unique_patients)
    print("MLP Done")
    current_results = results_rf + results_xgb + results_svr + results_mlp
    all_results.extend(current_results)
    df_current = pd.DataFrame(current_results)
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "RMSE_Mean": df_model["RMSE"].mean(),
            "MSE_Mean": df_model["MSE"].mean(),
            "MAE_Mean": df_model["MAE"].mean(),
        })
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)